# Regridding-resolution sensitivity of EERIE `pr` biases (1° vs 0.25°)

**Precipitation counterpart of `grid_resolution_sensitivity.ipynb`.** Same question — does the
*target regridding resolution* change the climatological bias we report? — but for **precipitation**
(`pr`), displayed in **mm/month**. Precipitation is even more resolution-sensitive than temperature:
orographic rainfall and coastal land–sea contrasts live at scales a 1° grid cannot hold.

We regrid each EERIE model (and ERA5) onto a **1°** grid and a **0.25°** grid, form the bias
`model − obs` on each, map where they disagree, auto-rank the largest-difference land cells, and
compare curated mountain/coastal boxes against the coarse **CMIP6** benchmark.

> **Units.** `pr` is stored internally as kg m⁻² s⁻¹ (= mm s⁻¹). We multiply by
> `86400 × 365.25/12` to display **mm/month** (mean monthly accumulation). Bias is linear, so
> converting the fields and converting the bias are equivalent.

> **Where to run.** Heavy (full-res EERIE `pr` + regridding) — DKRZ **compute node**, `feather` env.

In [ ]:
# Keep BLAS from spawning a thread per core (avoids RLIMIT_NPROC errors on DKRZ).
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import nereus as nr

from feather.config import FeatherConfig
from feather.data.cmor_loader import CMORLoader
from feather.data.obs import ObsLoader
from feather.data.cmip6 import CMIP6Loader
from feather.util.temporal import climatology, seasonal_climatology

plt.rcParams["figure.dpi"] = 110

## 1. Configuration & loaders

In [ ]:
CONFIG_PATH = "../configs/eerie.yaml"
VAR = "pr"
RES_HI, RES_LO = 0.25, 1.0            # target regridding resolutions (deg)
COARSEN = int(round(RES_LO / RES_HI)) # 0.25 -> 1.0 block factor (=4)
PR_TO_MMMONTH = 86400.0 * 365.25 / 12.0  # kg m-2 s-1 -> mm/month (mean month)

cfg = FeatherConfig.from_yaml(CONFIG_PATH)
PERIOD = cfg.get_period()
INFL = float(cfg.nereus.get("influence_radius", 80_000.0))

model_loader = CMORLoader(cfg)
obs_loader = ObsLoader(cfg)
cmip6_loader = CMIP6Loader(cfg) if cfg.cmip6.get("enabled", False) else None

EERIE_MODELS = list(cfg.models)
print("period :", PERIOD)
print("EERIE  :", EERIE_MODELS)
print("CMIP6  :", list(cmip6_loader.models) if cmip6_loader else "(disabled)")
print("mm/month factor: %.1f" % PR_TO_MMMONTH)

## 2. Regridding & unit helpers

`method="linear"`, config influence radius, 0..360 longitude frame. `to_mm_month` applies the
kg m⁻² s⁻¹ → mm/month conversion so every downstream field, bias, map and table is in mm/month.

In [ ]:
def regrid_field(field, lon, lat, resolution, influence_radius=INFL, method="linear"):
    """Regrid a scattered/rectilinear field to a regular `resolution`-deg grid."""
    vals = np.asarray(field).ravel()
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    if lon.ndim == 1 and lon.shape[0] != vals.shape[0]:
        lon, lat = np.meshgrid(lon, lat)
    _, interp = nr.regrid(
        vals, lon=np.asarray(lon), lat=np.asarray(lat),
        resolution=resolution, method=method,
        influence_radius=influence_radius,
        lon_bounds=(0.0, 360.0), as_xarray=True,
    )
    tlat = np.asarray(interp.target_lat)[:, 0]
    tlon = np.asarray(interp.target_lon)[0, :]
    return xr.DataArray(
        interp(vals), dims=("lat", "lon"),
        coords={"lat": tlat, "lon": tlon}, name=VAR,
    )


def to_common_1deg(bias_hi, bias_lo):
    """Block-mean the 0.25° bias to the 1° grid and align it with the 1° bias."""
    coarse = bias_hi.coarsen(lat=COARSEN, lon=COARSEN, boundary="trim").mean()
    return coarse.interp(lat=bias_lo.lat, lon=bias_lo.lon)


def to_mm_month(da):
    """kg m-2 s-1 -> mm/month."""
    return da * PR_TO_MMMONTH


def season_clim(da, period, season):
    """2-D climatology in mm/month for 'annual' | 'DJF' | 'MAM' | 'JJA' | 'SON'."""
    if season == "annual":
        c = climatology(da, period).compute()
    else:
        c = seasonal_climatology(da, period)[season].compute()
    return to_mm_month(c)

## 3. ERA5 reference climatology on both grids (mm/month)

In [ ]:
obs_da = obs_loader.load_for_model_var(VAR, PERIOD)          # ERA5 tp [kg m-2 s-1]
obs_clim = season_clim(obs_da, PERIOD, "annual")            # mm/month
obs_lat = obs_clim["lat" if "lat" in obs_clim.coords else "latitude"].values
obs_lon = obs_clim["lon" if "lon" in obs_clim.coords else "longitude"].values

obs_hi = regrid_field(obs_clim.values, obs_lon, obs_lat, RES_HI)
obs_lo = regrid_field(obs_clim.values, obs_lon, obs_lat, RES_LO)
print("ERA5 mm/month range: %.1f .. %.1f" % (float(obs_clim.min()), float(obs_clim.max())))

## 4. EERIE model climatologies & biases on both grids

In [ ]:
eerie = {}  # model -> dict of fields on the 1° grid (mm/month)
for m in EERIE_MODELS:
    try:
        da = model_loader.load_var(m, VAR, period=PERIOD, time_mean=False)
    except (KeyError, FileNotFoundError) as e:
        print(f"  skip {m}: {e}")
        continue
    clim = season_clim(da, PERIOD, "annual")
    mlat, mlon = clim["lat"].values, clim["lon"].values
    bias_hi = regrid_field(clim.values, mlon, mlat, RES_HI) - obs_hi
    bias_lo = regrid_field(clim.values, mlon, mlat, RES_LO) - obs_lo
    hi_on_lo = to_common_1deg(bias_hi, bias_lo)
    eerie[m] = {"bias_lo": bias_lo, "bias_hi_on_lo": hi_on_lo,
                "sensitivity": hi_on_lo - bias_lo}
    print(f"  {m}: done")
print("models with data:", list(eerie))

## 5. Sensitivity map (one model)

1° bias, 0.25° bias (coarsened to 1°), and their difference. `BrBG` (wet–dry) for the bias,
`PuOr_r` for the resolution sensitivity. Precipitation biases light up along orography and coasts.

In [ ]:
show = next(iter(eerie))
d = eerie[show]
fields = [("1° bias", d["bias_lo"], "BrBG", 100),
          ("0.25° bias (→ 1°)", d["bias_hi_on_lo"], "BrBG", 100),
          ("sensitivity (0.25° − 1°)", d["sensitivity"], "PuOr_r", 40)]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2),
                         subplot_kw={"projection": ccrs.Robinson()})
for ax, (ttl, fld, cmap, vm) in zip(axes, fields):
    p = ax.pcolormesh(fld.lon, fld.lat, fld, cmap=cmap, vmin=-vm, vmax=vm,
                      shading="auto", transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    ax.set_title(f"{show} — {ttl}", fontsize=10)
    fig.colorbar(p, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85,
                 label="mm/month")
fig.suptitle(f"pr bias vs ERA5 — regridding-resolution sensitivity ({PERIOD[0]}–{PERIOD[1]})",
             y=1.03)
plt.show()

## 6. Auto-identify the largest-difference regions

Mean `|sensitivity|` across EERIE models (land only); top 1° cells.

In [ ]:
# Land mask on the 1° grid from cartopy's land geometry.
import shapely.ops as sops
from shapely.geometry import Point
from shapely.prepared import prep
land_geom = prep(sops.unary_union(list(cfeature.LAND.geometries())))

sens_stack = xr.concat([v["sensitivity"] for v in eerie.values()], dim="model")
abs_sens = np.abs(sens_stack).mean("model")

lon2d, lat2d = np.meshgrid(abs_sens.lon.values, abs_sens.lat.values)
lon_pm = ((lon2d + 180) % 360) - 180
land = np.array([land_geom.contains(Point(x, y))
                 for x, y in zip(lon_pm.ravel(), lat2d.ravel())]).reshape(lat2d.shape)
abs_sens_land = abs_sens.where(land)

flat = abs_sens_land.stack(cell=("lat", "lon")).dropna("cell")
top = flat.sortby(flat, ascending=False)[:15]
print("Top land cells by mean |sensitivity| (mm/month):")
for c in top.cell.values:
    la, lo = c
    print(f"  lat {la:6.1f}  lon {lo:6.1f}   {float(top.sel(cell=c)):.2f} mm/month")

In [ ]:
fig = plt.figure(figsize=(12, 5))
ax = plt.axes(projection=ccrs.Robinson())
p = ax.pcolormesh(abs_sens_land.lon, abs_sens_land.lat, abs_sens_land,
                  cmap="magma_r", vmin=0, vmax=float(abs_sens_land.quantile(0.99)),
                  shading="auto", transform=ccrs.PlateCarree())
ax.coastlines(linewidth=0.4)
ax.set_title("Mean |0.25° − 1° pr-bias| across EERIE models (land)", fontsize=11)
fig.colorbar(p, ax=ax, orientation="vertical", shrink=0.8, label="mm/month")
plt.show()

## 7. Curated mountain & coastal regions

Same boxes as the `tas` notebook (lon in 0..360). Adjust from the auto-ranked cells above.

In [ ]:
REGIONS = {
    # name              (lat_min, lat_max, lon_min, lon_max, kind)
    "Alps":            (43, 48,   5,  16, "mountain"),
    "Tibetan Plateau": (27, 40,  75, 100, "mountain"),
    "Andes":           (-40, -18, 286, 296, "mountain"),
    "Rockies":         (35, 49,  244, 254, "mountain"),
    "Norwegian coast": (58, 68,   4,  14, "coastal"),
    "US West coast":   (34, 48,  234, 240, "coastal"),
    "Chilean coast":   (-40, -25, 284, 290, "coastal"),
    "New Zealand":     (-47, -34, 166, 179, "coastal"),
}

def box_mean(field, box):
    """cos(lat)-weighted mean over a lat/lon box; NaN cells are ignored."""
    lat0, lat1, lon0, lon1, _ = box
    sub = field.sel(lat=slice(lat0, lat1), lon=slice(lon0, lon1))
    w = np.cos(np.deg2rad(sub.lat))          # broadcasts over lon
    return float(sub.weighted(w).mean().values)

### Where are the regions? — boxes on ERA5 orography

ERA5 invariant geopotential (÷ g) regridded to 0.5° as a relief backdrop, with the curated boxes
drawn on top (**brown = mountain**, **blue = coastal**). This confirms the mountain boxes sit on the
Alps / Tibet / Andes / Rockies and the coastal boxes hug real coastlines.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

G = 9.80665  # standard gravity: geopotential (m2 s-2) -> height (m)
ERA5_OROG_GRIB = "/pool/data/ERA5/E5/sf/an/IV/129/E5sf00_IV_INVARIANT_129.grb"
try:
    _ds = xr.open_dataset(ERA5_OROG_GRIB, engine="cfgrib",
                          backend_kwargs={"indexpath": ""})
    _orog = _ds["z"] / G
    orog_map = regrid_field(_orog.values, _orog["longitude"].values,
                            _orog["latitude"].values, 0.5, influence_radius=60_000.0)
    have_orog = True
except Exception as e:  # noqa: BLE001
    print("ERA5 orography unavailable, using a plain land backdrop:", e)
    have_orog = False


def draw_region_boxes(ax, fontsize=8):
    kcol = {"mountain": "#7a3b12", "coastal": "#08306b"}
    for name, (lat0, lat1, lon0, lon1, kind) in REGIONS.items():
        ax.add_patch(mpatches.Rectangle(
            (lon0, lat0), lon1 - lon0, lat1 - lat0,
            transform=ccrs.PlateCarree(), fill=False,
            edgecolor=kcol[kind], linewidth=1.8, zorder=5))
        ax.text((lon0 + lon1) / 2.0, lat1 + 1.5, name,
                transform=ccrs.PlateCarree(), ha="center", va="bottom",
                fontsize=fontsize, color=kcol[kind], zorder=6,
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])


fig = plt.figure(figsize=(15, 7.5))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
ax.set_global()
if have_orog:
    p = ax.pcolormesh(orog_map.lon, orog_map.lat, orog_map, cmap="terrain",
                      vmin=0, vmax=4000, shading="auto", transform=ccrs.PlateCarree())
    fig.colorbar(p, ax=ax, orientation="horizontal", pad=0.05, shrink=0.6,
                 label="ERA5 orography [m]")
else:
    ax.add_feature(cfeature.LAND, facecolor="#dcdcc8")
    ax.add_feature(cfeature.OCEAN, facecolor="#c6dbef")
ax.coastlines(linewidth=0.5)
draw_region_boxes(ax)
ax.set_title("Curated mountain (brown) & coastal (blue) regions on ERA5 orography")
plt.show()

## 8. CMIP6 benchmark bias (MMM) on the 1° grid

Regrid each CMIP6 model to 1° (in mm/month), then average — the “regrid-then-average” MMM
convention. CMIP6 is coarse, so its 1° rendering is the fair benchmark column (see the companion
`cmip6_mmm_resolution_equivalence.ipynb` for why 0.25° would give the same thing).

In [ ]:
cmip6_bias_lo = None
if cmip6_loader is not None:
    members = []
    for cm in cmip6_loader.models:
        da = cmip6_loader.load_var("pr", cm, table="Amon", period=PERIOD, time_mean=True)
        if da is None:
            continue
        da = to_mm_month(da)
        clat = da["lat"].values if "lat" in da.coords else da["latitude"].values
        clon = da["lon"].values if "lon" in da.coords else da["longitude"].values
        try:
            reg = regrid_field(da.values, clon, clat, RES_LO,
                               influence_radius=max(INFL, 250_000.0))
        except Exception as e:  # noqa: BLE001
            print(f"  skip CMIP6 {cm}: {e}")
            continue
        members.append(reg - obs_lo)
    if members:
        cmip6_bias_lo = xr.concat(members, dim="model").mean("model")
        print(f"CMIP6 MMM from {len(members)} models")
print("CMIP6 benchmark available:", cmip6_bias_lo is not None)

## 9. Regional comparison table & bars (mm/month)

In [ ]:
import pandas as pd

ens_lo = xr.concat([v["bias_lo"] for v in eerie.values()], dim="model").mean("model")
ens_hi = xr.concat([v["bias_hi_on_lo"] for v in eerie.values()], dim="model").mean("model")

rows = []
for name, box in REGIONS.items():
    b_lo = box_mean(ens_lo, box)
    b_hi = box_mean(ens_hi, box)
    row = {"region": name, "kind": box[4],
           "EERIE 1°": b_lo, "EERIE 0.25°": b_hi,
           "shift (0.25−1°)": b_hi - b_lo}
    row["CMIP6 MMM"] = box_mean(cmip6_bias_lo, box) if cmip6_bias_lo is not None else np.nan
    row["|EERIE0.25−CMIP6|"] = abs(b_hi - row["CMIP6 MMM"])
    rows.append(row)

df = pd.DataFrame(rows).set_index("region").round(2)
df

In [ ]:
x = np.arange(len(df))
w = 0.27
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w, df["EERIE 1°"], w, label="EERIE 1°", color="#8da0cb")
ax.bar(x, df["EERIE 0.25°"], w, label="EERIE 0.25°", color="#1f77b4")
ax.bar(x + w, df["CMIP6 MMM"], w, label="CMIP6 MMM", color="#999999")
ax.axhline(0, color="k", lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n({k})" for n, k in zip(df.index, df["kind"])],
                   rotation=30, ha="right", fontsize=9)
ax.set_ylabel("pr bias vs ERA5 [mm/month]")
ax.set_title("Regional pr bias: regridding resolution vs CMIP6 benchmark")
ax.legend()
fig.tight_layout()
plt.show()

## 10. Seasonal variant (DJF & JJA)

`compute_bias_set(season)` reruns the whole pipeline for one season (fields in mm/month) and
returns the 1°-grid fields. Reuses the `land` mask (§6), `REGIONS` and `box_mean` (§7).

In [ ]:
def compute_bias_set(season):
    """All 1°-grid pr bias fields (mm/month) for one season."""
    o = season_clim(obs_da, PERIOD, season)
    olat = o["lat" if "lat" in o.coords else "latitude"].values
    olon = o["lon" if "lon" in o.coords else "longitude"].values
    o_hi = regrid_field(o.values, olon, olat, RES_HI)
    o_lo = regrid_field(o.values, olon, olat, RES_LO)

    ee = {}
    for m in EERIE_MODELS:
        try:
            da = model_loader.load_var(m, VAR, period=PERIOD, time_mean=False)
        except (KeyError, FileNotFoundError):
            continue
        c = season_clim(da, PERIOD, season)
        b_hi = regrid_field(c.values, c["lon"].values, c["lat"].values, RES_HI) - o_hi
        b_lo = regrid_field(c.values, c["lon"].values, c["lat"].values, RES_LO) - o_lo
        hol = to_common_1deg(b_hi, b_lo)
        ee[m] = {"bias_lo": b_lo, "bias_hi_on_lo": hol, "sensitivity": hol - b_lo}

    c6 = None
    if cmip6_loader is not None:
        mem = []
        for cm in cmip6_loader.models:
            da = cmip6_loader.load_var(
                "pr", cm, table="Amon", period=PERIOD,
                season=None if season == "annual" else season, time_mean=True,
            )
            if da is None:
                continue
            da = to_mm_month(da)
            clat = da["lat"].values if "lat" in da.coords else da["latitude"].values
            clon = da["lon"].values if "lon" in da.coords else da["longitude"].values
            try:
                reg = regrid_field(da.values, clon, clat, RES_LO,
                                   influence_radius=max(INFL, 250_000.0))
            except Exception:  # noqa: BLE001
                continue
            mem.append(reg - o_lo)
        if mem:
            c6 = xr.concat(mem, dim="model").mean("model")

    ens_lo = xr.concat([v["bias_lo"] for v in ee.values()], dim="model").mean("model")
    ens_hi = xr.concat([v["bias_hi_on_lo"] for v in ee.values()], dim="model").mean("model")
    abs_sens = np.abs(
        xr.concat([v["sensitivity"] for v in ee.values()], dim="model")
    ).mean("model")
    return {"eerie": ee, "cmip6": c6, "ens_lo": ens_lo,
            "ens_hi": ens_hi, "abs_sens": abs_sens}


SEASON_SETS = {s: compute_bias_set(s) for s in ["DJF", "JJA"]}
print("computed seasons:", list(SEASON_SETS))

In [ ]:
fig, axes = plt.subplots(1, len(SEASON_SETS), figsize=(7 * len(SEASON_SETS), 4.2),
                         subplot_kw={"projection": ccrs.Robinson()})
axes = np.atleast_1d(axes)
for ax, (s, S) in zip(axes, SEASON_SETS.items()):
    fld = S["abs_sens"].where(land)          # `land` mask from §6
    p = ax.pcolormesh(fld.lon, fld.lat, fld, cmap="magma_r",
                      vmin=0, vmax=float(fld.quantile(0.99)),
                      shading="auto", transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    ax.set_title(f"{s}: mean |0.25° − 1° pr-bias| (land)", fontsize=10)
    fig.colorbar(p, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85, label="mm/month")
fig.suptitle("Seasonal regridding-resolution sensitivity (pr)", y=1.02)
plt.show()

In [ ]:
import pandas as pd

seas_rows = []
for s, S in SEASON_SETS.items():
    for name, box in REGIONS.items():
        b_lo = box_mean(S["ens_lo"], box)
        b_hi = box_mean(S["ens_hi"], box)
        c6 = box_mean(S["cmip6"], box) if S["cmip6"] is not None else np.nan
        seas_rows.append({
            "season": s, "region": name, "kind": box[4],
            "EERIE 1°": b_lo, "EERIE 0.25°": b_hi,
            "shift (0.25−1°)": b_hi - b_lo, "CMIP6 MMM": c6,
            "|EERIE0.25−CMIP6|": abs(b_hi - c6),
        })

df_seas = pd.DataFrame(seas_rows).set_index(["season", "region"]).round(2)
df_seas

## 11. Takeaways

- Precipitation sensitivity to the 1°-vs-0.25° choice is concentrated over **orography**
  (windward/leeward rainfall gradients) and **coasts** (land–sea rain contrasts) — expect larger
  mm/month shifts than the `tas` case.
- Large `shift (0.25−1°)` and large `|EERIE0.25−CMIP6|` in mountain/coastal boxes quantify the
  precipitation structure that only the high-resolution grid retains.
- Seasonally, monsoon and storm-track regions dominate — compare DJF vs JJA per region.

*Units are mm/month throughout.* Swap `PR_TO_MMMONTH` for `86400` to display mm/day instead.